In [ ]:
# !pip uninstall -y numpy matplotlib scikit-learn imbalanced-learn
# !pip install numpy==1.23.5 matplotlib==3.6.3 scikit-learn==1.1.3 imbalanced-learn==0.10.1

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: matplotlib 3.7.1
Uninstalling matplotlib-3.7.1:
  Successfully uninstalled matplotlib-3.7.1
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Found existing installation: imbalanced-learn 0.12.3
Uninstalling imbalanced-learn-0.12.3:
  Successfully uninstalled imbalanced-learn-0.12.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.5/30.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.0/226.0 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore

In [ ]:
import os
import time
import pickle

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fftpack import dct

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE

In [ ]:


# Replace 'your_file.csv' with the actual path to your CSV file
file_path = r"/kaggle/input/tdid-kaggle/TDID_kaggle/labels1.csv"

column_names = ['Image', 'Stage_1', 'Stage_2']

df = pd.read_csv(file_path, header=None, names=column_names)

df=df[1:481]

print(df)

     Image Stage_1 Stage_2
1      1_1       0       0
2      1_2       0       0
3      2_1       1       2
4      3_1       2       4
5      4_1       2       4
..     ...     ...     ...
476  396_1       0       0
477  397_1       1       2
478  398_1       2       4
479  399_1       2       4
480  400_1       2       4

[480 rows x 3 columns]


In [ ]:


# Define the root directory
rootdir = r"/kaggle/input/tdid-kaggle/TDID_kaggle/datapre"
image_path = [os.path.join(rootdir, file) for file in os.listdir(rootdir)]

# Function to generate zigzag order indices
def zigzag_indices(n):
    index_order = sorted(((x, y) for x in range(n) for y in range(n)),
                         key=lambda s: (s[0] + s[1], s[1] if (s[0] + s[1]) % 2 else -s[1]))
    return [(x, y) for x, y in index_order]

# Function to extract localized DCT descriptors
def extract_localized_dct_descriptors(image):
    # Resize image and convert to grayscale
    resized_image = cv2.resize(image, (256, 256))
    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
    height, width = gray_image.shape
    descriptors = []
    zigzag_order = zigzag_indices(8)

    # Apply localized DCT in 8x8 blocks
    for i in range(0, height, 8):
        for j in range(0, width, 8):
            cell = gray_image[i:i + 8, j:j + 8]
            
            # Apply DCT
            dct_cell = cv2.dct(np.float32(cell))
            
            # Keep first 36 coefficients in zigzag order
            cell_descriptor = [dct_cell[idx] for idx in zigzag_order[:36]]
            
            # Compute average
            avg_value = np.mean(cell_descriptor)
            
            # Binary thresholding
            binary_descriptor = [1 if coeff > avg_value else 0 for coeff in cell_descriptor]
            
            # Convert binary to decimal
            decimal_value = int("".join(map(str, binary_descriptor)), 2)
            
            descriptors.append(decimal_value)

    return np.array(descriptors)


# Define a global dataframe (UPDATED)
dct_df = pd.DataFrame(columns=['image_name', 'features', 'labels_1', 'labels_2'])

def main():
    global dct_df
    t0 = time.time()

    for file in image_path:
        img = cv2.imread(file)
        
        # Extract image name
        img_name = os.path.basename(file).split('.')[0]
        
        # Extract labels from df (assumes df is already defined)
        label_1 = df.loc[df['Image'] == img_name, 'Stage_1'].iloc[0]
        label_2 = df.loc[df['Image'] == img_name, 'Stage_2'].iloc[0]
        
        # Apply localized DCT
        img_des = extract_localized_dct_descriptors(img)
        
        if img_des is not None:
            new_data = pd.DataFrame({
                'image_name': [img_name],   # ✅ Added this line
                'features': [img_des],
                'labels_1': [label_1],
                'labels_2': [label_2]
            })
            
            dct_df = pd.concat([dct_df, new_data], ignore_index=True)
    
    print('Time taken: {:.2f} seconds'.format(time.time() - t0))


print('\nCalculating')
main()

print(dct_df)


Calculating
Time taken: 16.58 seconds
    image_name                                           features labels_1  \
0        335_1  [36936552832, 36994828200, 38693788100, 430044...        1   
1        261_1  [39413424354, 37302157425, 43929457156, 480363...        2   
2        341_1  [62697963904, 36642524610, 34813805464, 368773...        2   
3        366_1  [34968307776, 35971399680, 35971399812, 447857...        1   
4         15_1  [36927514160, 49509828619, 34662074928, 408406...        2   
..         ...                                                ...      ...   
344      289_1  [51422365264, 44221207625, 41905520700, 443464...        2   
345      294_3  [35857968706, 36373209650, 43012993056, 412426...        2   
346       83_1  [41561105454, 45696270336, 48077212180, 351933...        2   
347      156_2  [45524975872, 43243963864, 43001851800, 347291...        2   
348      186_1  [41023447810, 47617002560, 47314370828, 507214...        2   

    labels_2  
0        

In [5]:
print("Length of features in hog df", len(dct_df['features'].iloc[-1]))

Length of features in hog df 1024


In [6]:
dct_df

,image_name,features,labels_1,labels_2
0,335_1,"[36936552832, 36994828200, 38693788100, 430044...",1,2
1,261_1,"[39413424354, 37302157425, 43929457156, 480363...",2,4
2,341_1,"[62697963904, 36642524610, 34813805464, 368773...",2,4
3,366_1,"[34968307776, 35971399680, 35971399812, 447857...",1,3
4,15_1,"[36927514160, 49509828619, 34662074928, 408406...",2,4
...,...,...,...,...
344,289_1,"[51422365264, 44221207625, 41905520700, 443464...",2,4
345,294_3,"[35857968706, 36373209650, 43012993056, 412426...",2,5
346,83_1,"[41561105454, 45696270336, 48077212180, 351933...",2,4
347,156_2,"[45524975872, 43243963864, 43001851800, 347291...",2,4


In [7]:

# Dropping the 'labels_1' column
dct_df_stage_1 = dct_df.drop(columns=['labels_2'])
# Dropping the 'labels_1' column
dct_df_stage_2 = dct_df.drop(columns=['labels_1'])



In [8]:
# Display the updated DataFrame
print(dct_df_stage_1)

    image_name                                           features labels_1
0        335_1  [36936552832, 36994828200, 38693788100, 430044...        1
1        261_1  [39413424354, 37302157425, 43929457156, 480363...        2
2        341_1  [62697963904, 36642524610, 34813805464, 368773...        2
3        366_1  [34968307776, 35971399680, 35971399812, 447857...        1
4         15_1  [36927514160, 49509828619, 34662074928, 408406...        2
..         ...                                                ...      ...
344      289_1  [51422365264, 44221207625, 41905520700, 443464...        2
345      294_3  [35857968706, 36373209650, 43012993056, 412426...        2
346       83_1  [41561105454, 45696270336, 48077212180, 351933...        2
347      156_2  [45524975872, 43243963864, 43001851800, 347291...        2
348      186_1  [41023447810, 47617002560, 47314370828, 507214...        2

[349 rows x 3 columns]


In [9]:
# Display the updated DataFrame
print(dct_df_stage_2)

    image_name                                           features labels_2
0        335_1  [36936552832, 36994828200, 38693788100, 430044...        2
1        261_1  [39413424354, 37302157425, 43929457156, 480363...        4
2        341_1  [62697963904, 36642524610, 34813805464, 368773...        4
3        366_1  [34968307776, 35971399680, 35971399812, 447857...        3
4         15_1  [36927514160, 49509828619, 34662074928, 408406...        4
..         ...                                                ...      ...
344      289_1  [51422365264, 44221207625, 41905520700, 443464...        4
345      294_3  [35857968706, 36373209650, 43012993056, 412426...        5
346       83_1  [41561105454, 45696270336, 48077212180, 351933...        4
347      156_2  [45524975872, 43243963864, 43001851800, 347291...        4
348      186_1  [41023447810, 47617002560, 47314370828, 507214...        5

[349 rows x 3 columns]


In [ ]:


# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 2 or 3
dct_df_benign = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([2, 3])]

# Display the filtered dataframe
print(dct_df_benign)


    image_name                                           features  labels_2
0        335_1  [36936552832, 36994828200, 38693788100, 430044...         2
3        366_1  [34968307776, 35971399680, 35971399812, 447857...         3
8         31_1  [36911319337, 36890085257, 36676703737, 347557...         3
13       364_1  [34398733348, 52614398208, 52613349376, 526804...         2
18       378_1  [45251371018, 47264831520, 47451217172, 451511...         2
..         ...                                                ...       ...
299      303_1  [66991433296, 48373198964, 48654727170, 530666...         2
302      380_1  [63782913128, 38844256791, 64713947236, 368931...         3
313      397_1  [36784365467, 36853186984, 36680639448, 344563...         2
317        2_1  [37598604835, 39900752580, 65804563022, 667817...         2
327      329_1  [35857151514, 54086541321, 49409237072, 420441...         2

[61 rows x 3 columns]


In [ ]:


# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 4 or 5
dct_df_malignant = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([4,5])]

# Display the filtered dataframe
print(dct_df_malignant)


    image_name                                           features  labels_2
1        261_1  [39413424354, 37302157425, 43929457156, 480363...         4
2        341_1  [62697963904, 36642524610, 34813805464, 368773...         4
4         15_1  [36927514160, 49509828619, 34662074928, 408406...         4
5        379_1  [40819834498, 47517446192, 47371519488, 475891...         4
6         65_1  [36919631289, 38928712096, 38923242384, 365959...         4
..         ...                                                ...       ...
344      289_1  [51422365264, 44221207625, 41905520700, 443464...         4
345      294_3  [35857968706, 36373209650, 43012993056, 412426...         5
346       83_1  [41561105454, 45696270336, 48077212180, 351933...         4
347      156_2  [45524975872, 43243963864, 43001851800, 347291...         4
348      186_1  [41023447810, 47617002560, 47314370828, 507214...         5

[288 rows x 3 columns]


# Stage 1 Benign/Malignant


In [ ]:
##only train oversampled 



# =====================================================
# Ensure labels are int
# =====================================================

dct_df_stage_1['labels_1'] = (
    dct_df_stage_1['labels_1'].astype(int)
)

# =====================================================
# Separate features, labels, names
# =====================================================

X = np.array(
    dct_df_stage_1['features'].tolist()
)

y = np.array(
    dct_df_stage_1['labels_1']
)

names = np.array(
    dct_df_stage_1['image_name']
)

# =====================================================
# Kaggle output root
# =====================================================

output_root = "/kaggle/working"

model_dir = os.path.join(
    output_root,
    "models_stage_1"
)

result_dir = os.path.join(
    output_root,
    "results_stage_1"
)

os.makedirs(model_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

# =====================================================
# Parameters
# =====================================================

num_splits = 5
kernel = 'rbf'

# =====================================================
# Metric storage
# =====================================================

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

# =====================================================
# Stratified KFold
# =====================================================

kfold = StratifiedKFold(
    n_splits=num_splits,
    shuffle=True
)

# =====================================================
# Store predictions
# =====================================================

all_results = pd.DataFrame(
    columns=[
        'fold',
        'image_name',
        'true_label',
        'predicted_label'
    ]
)

# =====================================================
# Cross-validation
# =====================================================

for fold, (train_index, test_index) in enumerate(
        kfold.split(X, y), start=1):

    print("\n" + "="*50)
    print(f"Fold {fold}")
    print("="*50)

    # =================================================
    # Split data
    # =================================================

    X_train = X[train_index]
    X_test  = X[test_index]

    y_train = y[train_index]
    y_test  = y[test_index]

    names_test = names[test_index]

    # =================================================
    # Original Test Distribution
    # =================================================

    unique_test, counts_test = np.unique(
        y_test,
        return_counts=True
    )

    print("\nOriginal Test Distribution:")

    for cls, cnt in zip(unique_test, counts_test):
        print(f"Class {cls}: {cnt}")

    print(f"\nTotal Test Samples: {len(y_test)}")

    # =================================================
    # SMOTE on TRAIN
    # =================================================

    smote_train = SMOTE(random_state=5)

    X_train_smote, y_train_smote = (
        smote_train.fit_resample(
            X_train,
            y_train
        )
    )

    # =================================================
    # Training Distribution After SMOTE
    # =================================================

    print("\nTraining Distribution After SMOTE:")

    unique_sm, counts_sm = np.unique(
        y_train_smote,
        return_counts=True
    )

    for cls, cnt in zip(unique_sm, counts_sm):
        print(f"Class {cls}: {cnt}")

    X_test_final = X_test
    y_test_final = y_test

    # =================================================
    # TRAIN SVM
    # =================================================

    svm = SVC(
        kernel=kernel
    )

    svm.fit(
        X_train_smote,
        y_train_smote
    )

    # =================================================
    # PREDICT
    # =================================================

    y_pred = svm.predict(
        X_test_final
    )

    # =================================================
    # Save model
    # =================================================

    model_path = os.path.join(
        model_dir,
        f"svm_fold_{fold}.pkl"
    )

    with open(model_path, 'wb') as f:
        pickle.dump(svm, f)

    # =================================================
    # Save fold predictions
    # =================================================

    fold_df = pd.DataFrame({
        'fold': [fold] * len(y_test_final),
        'image_name': ['mixed_test']
                      * len(y_test_final),
        'true_label': y_test_final,
        'predicted_label': y_pred
    })

    all_results = pd.concat(
        [all_results, fold_df],
        ignore_index=True
    )

    # =================================================
    # Confusion Matrix
    # =================================================

    cm = confusion_matrix(
        y_test_final,
        y_pred,
    )

    tn, fp, fn, tp = cm.ravel()

    # =================================================
    # Metrics
    # =================================================

    specificity = (
        tn / (tn + fp + 1e-10)
    )

    sensitivity = (
        tp / (tp + fn + 1e-10)
    )

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + 1e-10)
    )

    precision = (
        tp / (tp + fp + 1e-10)
    )

    f1_score = (
        2 * (precision * sensitivity)
    ) / (
        precision + sensitivity + 1e-10
    )

    # =================================================
    # Store metrics
    # =================================================

    specificity_scores.append(
        specificity
    )

    sensitivity_scores.append(
        sensitivity
    )

    accuracy_scores.append(
        accuracy
    )

    precision_scores.append(
        precision
    )

    f1_scores.append(
        f1_score
    )

# =====================================================
# SAVE CSV
# =====================================================

csv_path = os.path.join(
    result_dir,
    "fold_predictions.csv"
)

all_results.to_csv(
    csv_path,
    index=False
)

# =====================================================
# Average Metrics
# =====================================================

average_accuracy = np.mean(
    accuracy_scores
)

average_specificity = np.mean(
    specificity_scores
)

average_sensitivity = np.mean(
    sensitivity_scores
)

average_precision = np.mean(
    precision_scores
)

average_f1_score = np.mean(
    f1_scores
)

# =====================================================
# FINAL RESULTS
# =====================================================

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print("\nAverage Precision:",
      average_precision)

print("Average F1 Score:",
      average_f1_score)

print("\nAverage Specificity:",
      average_specificity)

print("Average Sensitivity (Recall):",
      average_sensitivity)

print("Average Accuracy:",
      average_accuracy)


Fold 1

Original Test Distribution:
Class 1: 13
Class 2: 57

Total Test Samples: 70

Training Distribution After SMOTE:
Class 1: 231
Class 2: 231

Fold 2

Original Test Distribution:
Class 1: 12
Class 2: 58

Total Test Samples: 70

Training Distribution After SMOTE:
Class 1: 230
Class 2: 230

Fold 3

Original Test Distribution:
Class 1: 12
Class 2: 58

Total Test Samples: 70

Training Distribution After SMOTE:
Class 1: 230
Class 2: 230

Fold 4

Original Test Distribution:
Class 1: 12
Class 2: 58

Total Test Samples: 70

Training Distribution After SMOTE:
Class 1: 230
Class 2: 230

Fold 5

Original Test Distribution:
Class 1: 12
Class 2: 57

Total Test Samples: 69

Training Distribution After SMOTE:
Class 1: 231
Class 2: 231

FINAL RESULTS

Average Precision: 0.8068723451810739
Average F1 Score: 0.893044871744063

Average Specificity: 0.7602540834832537
Average Sensitivity (Recall): 0.9999999999982638
Average Accuracy: 0.8801270417415228
